# 🧠 LSTM for RUL Prediction - Comprehensive Tutorial

## Learning Objectives

By the end of this tutorial, you will:
- Understand LSTM architecture in detail (gates, cell state, hidden state)
- Learn why LSTM is ideal for RUL prediction
- Build multiple LSTM model designs with different architectures
- Compare different LSTM configurations (1-layer, 2-layer, 3-layer, bidirectional)
- Visualize LSTM cell structure and data flow
- Apply LSTMs to NASA turbofan engine RUL prediction
- Understand how different LSTM designs affect performance

---

## What is LSTM (Long Short-Term Memory)?

**LSTM** is a special type of RNN designed to solve the **vanishing gradient problem** and remember information for long periods.

### Why LSTM for RUL Prediction?

- **Long-term Memory**: Engine degradation happens over many cycles
- **Gated Architecture**: Controls what to remember and forget
- **Temporal Patterns**: Captures degradation trends over time
- **Stable Training**: Less prone to vanishing gradients than SimpleRNN

### Key Innovation: Gated Architecture

LSTMs use **gates** to control information flow:
- **Forget Gate**: Decides what information to discard from cell state
- **Input Gate**: Decides what new information to store
- **Output Gate**: Decides what information to output

### Our Goal:

Build **multiple LSTM architectures** to predict RUL using NASA turbofan engine data and compare their performance!

---

## Step 1: Import Required Libraries

In [ ]:
# Import libraries for data handling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch, FancyBboxPatch, Arrow
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import machine learning components
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow/Keras for LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set visualization style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')

sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load NASA Turbofan Engine Data in

In [ ]:
# Define data path
data_path = Path('dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = [
    'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
    'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32'
]
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data from CSV files"""
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL (Remaining Useful Life) for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data for FD001 dataset
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print("✅ NASA Turbofan Engine Data loaded successfully!")
print(f"\\n📊 Training data shape: {train_df.shape}")
print(f"📊 Test data shape: {test_df.shape}")
print(f"🚁 Number of engines in training: {train_df['unit'].nunique()}")
print(f"🚁 Number of engines in test: {test_df['unit'].nunique()}")
print(f"⏱️  RUL range in training: {train_df['RUL'].min()} to {train_df['RUL'].max()} cycles")
print(f"📈 Total cycles in training: {train_df['time'].sum():,}")

## Step 3: Visualize LSTM Cell Structure in Detail

Let's understand how LSTM cells work with detailed visualizations!

In [ ]:
# Detailed LSTM Cell Visualization
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.set_xlim(-1, 8)
ax.set_ylim(-1, 7)
ax.axis('off')
ax.set_title('LSTM Cell Structure - Detailed View\\nUnderstanding Gates and Information Flow', 
             fontsize=18, fontweight='bold', pad=25)

# Main LSTM cell box
cell_x, cell_y = 3.5, 3.5
cell_width, cell_height = 3, 4

rect = Rectangle((cell_x - cell_width/2, cell_y - cell_height/2), cell_width, cell_height,
                fill=False, edgecolor='black', linewidth=3)
ax.add_patch(rect)
ax.text(cell_x, cell_y + cell_height/2 + 0.3, 'LSTM Cell', ha='center', fontsize=14, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', alpha=0.8))

# Inputs (left side)
circle_x = Circle((0.5, cell_y), 0.25, color='#4A90E2', ec='black', lw=2)
ax.add_patch(circle_x)
ax.text(0.5, cell_y, 'X_t', ha='center', va='center', fontsize=12, fontweight='bold')

circle_h_prev = Circle((0.5, cell_y + 1.5), 0.25, color='#FFA500', ec='black', lw=2)
ax.add_patch(circle_h_prev)
ax.text(0.5, cell_y + 1.5, 'H_{t-1}', ha='center', va='center', fontsize=11, fontweight='bold')

circle_c_prev = Circle((0.5, cell_y - 1.5), 0.25, color='#9B59B6', ec='black', lw=2)
ax.add_patch(circle_c_prev)
ax.text(0.5, cell_y - 1.5, 'C_{t-1}', ha='center', va='center', fontsize=11, fontweight='bold')

# Gates (inside cell)
gate_positions = [
    (cell_x, cell_y + 1.2, 'Forget\\nGate', '#E74C3C'),  # Red
    (cell_x, cell_y + 0.3, 'Input\\nGate', '#3498DB'),   # Blue
    (cell_x, cell_y - 0.6, 'Output\\nGate', '#2ECC71')   # Green
]

for x, y, name, color in gate_positions:
    rect_gate = Rectangle((x - 0.7, y - 0.3), 1.4, 0.6,
                         facecolor=color, edgecolor='black', linewidth=2, alpha=0.7)
    ax.add_patch(rect_gate)
    ax.text(x, y, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Cell state (inside, horizontal)
rect_c_state = Rectangle((cell_x - 0.6, cell_y - 1.1), 1.2, 0.3,
                        fill=False, edgecolor='purple', linewidth=2.5, linestyle='-')
ax.add_patch(rect_c_state)
ax.text(cell_x, cell_y - 1.3, 'Cell State (C_t)', ha='center', fontsize=9, style='italic', 
       color='purple', fontweight='bold')

# Tanh activation (for candidate values)
rect_tanh = Rectangle((cell_x - 0.5, cell_y - 0.1), 1.0, 0.3,
                    facecolor='#F39C12', edgecolor='black', linewidth=1.5, alpha=0.6)
ax.add_patch(rect_tanh)
ax.text(cell_x, cell_y + 0.05, 'tanh', ha='center', va='center', fontsize=9, fontweight='bold')

# Outputs (right side)
circle_h = Circle((7, cell_y + 1.5), 0.25, color='#50C878', ec='black', lw=2)
ax.add_patch(circle_h)
ax.text(7, cell_y + 1.5, 'H_t', ha='center', va='center', fontsize=12, fontweight='bold')

circle_c = Circle((7, cell_y - 1.5), 0.25, color='#9B59B6', ec='black', lw=2)
ax.add_patch(circle_c)
ax.text(7, cell_y - 1.5, 'C_t', ha='center', va='center', fontsize=12, fontweight='bold')

circle_y = Circle((7, cell_y), 0.25, color='#FF6B6B', ec='black', lw=2)
ax.add_patch(circle_y)
ax.text(7, cell_y, 'Y_t', ha='center', va='center', fontsize=12, fontweight='bold')

# Connections - Input to gates
for gate_y in [cell_y + 1.2, cell_y + 0.3, cell_y - 0.6]:
    ax.arrow(0.75, cell_y, 2.0, gate_y - cell_y, head_width=0.12, head_length=0.15, 
            fc='blue', ec='blue', linewidth=1.5, alpha=0.6)

# Previous hidden to gates
for gate_y in [cell_y + 1.2, cell_y + 0.3, cell_y - 0.6]:
    ax.arrow(0.75, cell_y + 1.5, 2.0, gate_y - (cell_y + 1.5), head_width=0.12, head_length=0.15, 
            fc='orange', ec='orange', linewidth=1.5, alpha=0.6)

# Previous cell state to forget gate and cell state
ax.arrow(0.75, cell_y - 1.5, 2.0, 2.7, head_width=0.12, head_length=0.15, 
        fc='purple', ec='purple', linewidth=2.5, linestyle='--', alpha=0.8)
ax.arrow(0.75, cell_y - 1.5, 2.0, 0.4, head_width=0.12, head_length=0.15, 
        fc='purple', ec='purple', linewidth=2.5, linestyle='--', alpha=0.8)

# Gate outputs to cell state
ax.arrow(cell_x, cell_y + 1.2, 0, -0.9, head_width=0.15, head_length=0.1, 
        fc='red', ec='red', linewidth=2, alpha=0.7)
ax.arrow(cell_x, cell_y + 0.3, 0, -0.9, head_width=0.15, head_length=0.1, 
        fc='blue', ec='blue', linewidth=2, alpha=0.7)

# Cell state to output gate and output
ax.arrow(cell_x, cell_y - 0.9, 0, 0.3, head_width=0.15, head_length=0.1, 
        fc='purple', ec='purple', linewidth=2.5, linestyle='--', alpha=0.8)
ax.arrow(cell_x + 0.6, cell_y - 0.9, 2.4, 0.6, head_width=0.12, head_length=0.15, 
        fc='purple', ec='purple', linewidth=2.5, linestyle='--', alpha=0.8)

# Output gate to hidden state
ax.arrow(cell_x, cell_y - 0.6, 0, 2.1, head_width=0.15, head_length=0.1, 
        fc='green', ec='green', linewidth=2, alpha=0.7)
ax.arrow(cell_x + 0.6, cell_y + 1.5, 2.4, 0, head_width=0.12, head_length=0.15, 
        fc='green', ec='green', linewidth=2, alpha=0.7)

# Labels
ax.text(0.5, cell_y - 2.2, 'Inputs', ha='center', fontsize=12, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black', alpha=0.7))
ax.text(7, cell_y - 2.2, 'Outputs', ha='center', fontsize=12, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightgreen', edgecolor='black', alpha=0.7))

# Legend
legend_elements = [
    plt.Line2D([0], [0], color='blue', lw=3, label='Input (X_t)'),
    plt.Line2D([0], [0], color='orange', lw=3, label='Previous Hidden (H_{t-1})'),
    plt.Line2D([0], [0], color='purple', lw=3, linestyle='--', label='Cell State (C_t)'),
    plt.Line2D([0], [0], color='red', lw=3, label='Forget Gate'),
    plt.Line2D([0], [0], color='#3498DB', lw=3, label='Input Gate'),
    plt.Line2D([0], [0], color='green', lw=3, label='Output Gate')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.show()

print("✅ Detailed LSTM cell structure visualization created!")
print("\\n💡 LSTM Components Explained:")
print("   - Forget Gate (Red): Decides what to forget from C_{t-1}")
print("   - Input Gate (Blue): Decides what new info to add to cell state")
print("   - Output Gate (Green): Decides what to output based on cell state")
print("   - Cell State (Purple): Long-term memory that flows through time")
print("   - Hidden State (Orange/Green): Short-term memory/output")

In [ ]:
# Visualize LSTM unrolled through time for RUL prediction
fig, ax = plt.subplots(1, 1, figsize=(18, 8))
ax.set_xlim(-0.5, 10)
ax.set_ylim(-0.5, 5)
ax.axis('off')
ax.set_title('LSTM Unrolled Through Time for RUL Prediction\\nHow Sequences are Processed', 
             fontsize=16, fontweight='bold', pad=20)

# Time steps
time_steps = ['t-2', 't-1', 't']
colors = ['#FFA500', '#50C878', '#4A90E2']

for t_idx, (t_label, color) in enumerate(zip(time_steps, colors)):
    x_offset = t_idx * 3.2
    
    # Sensor inputs at this timestep
    for i in range(2):
        circle = Circle((x_offset, 0.5 + i*0.6), 0.12, color='#4A90E2', ec='black', lw=1.2)
        ax.add_patch(circle)
        if i == 0:
            ax.text(x_offset, 0.5 + i*0.6, f'S{t_label}', ha='center', va='center', 
                   fontsize=8, fontweight='bold')
    
    # LSTM cell
    rect = Rectangle((x_offset - 0.4, 2), 0.8, 1.2, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_offset, 2.6, 'LSTM', ha='center', va='center', fontsize=9, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    
    # Hidden state
    circle_h = Circle((x_offset + 1.2, 2.6), 0.15, color=color, ec='black', lw=1.5)
    ax.add_patch(circle_h)
    ax.text(x_offset + 1.2, 2.6, f'H{t_label}', ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Cell state
    circle_c = Circle((x_offset + 1.2, 1.8), 0.15, color='purple', ec='black', lw=1.5)
    ax.add_patch(circle_c)
    ax.text(x_offset + 1.2, 1.8, f'C{t_label}', ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Connections within timestep
    ax.plot([x_offset, x_offset], [1.7, 2], 'gray', alpha=0.4, linewidth=1.2)
    ax.plot([x_offset + 0.4, x_offset + 1.05], [2.6, 2.6], 'gray', alpha=0.4, linewidth=1.2)
    ax.plot([x_offset + 0.4, x_offset + 1.05], [1.8, 1.8], 'purple', alpha=0.6, linewidth=1.5, linestyle='--')
    
    # Recurrent connections (hidden and cell state to next timestep)
    if t_idx < len(time_steps) - 1:
        ax.arrow(x_offset + 1.35, 2.6, 1.5, 0, head_width=0.1, head_length=0.2, 
                fc=color, ec=color, linewidth=2, alpha=0.7)
        ax.arrow(x_offset + 1.35, 1.8, 1.5, 0, head_width=0.1, head_length=0.2, 
                fc='purple', ec='purple', linewidth=2.5, linestyle='--', alpha=0.7)
        if t_idx == 0:
            ax.text(x_offset + 2.1, 2.8, 'Memory', ha='center', fontsize=8, color='red', fontweight='bold')

# RUL output (at last timestep)
circle_rul = Circle((9.5, 2.2), 0.2, color='#FF6B6B', ec='black', lw=2)
ax.add_patch(circle_rul)
ax.text(9.5, 2.2, 'RUL', ha='center', va='center', fontsize=11, fontweight='bold')

# Connection from last hidden to RUL
ax.arrow(9.2, 2.6, 0.1, -0.2, head_width=0.12, head_length=0.15, 
        fc='green', ec='green', linewidth=2.5, alpha=0.8)

# Labels
ax.text(4.8, -0.3, 'Time Steps →', ha='center', fontsize=12, fontweight='bold', style='italic')
ax.text(0, 1.1, 'Sensor\\nSequences', ha='center', fontsize=10, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
ax.text(9.5, 1.5, 'RUL\\nPrediction', ha='center', fontsize=10, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

plt.tight_layout()
plt.show()

print("✅ LSTM unrolled visualization created!")
print("\\n💡 How LSTM Processes RUL Sequences:")
print("   - Each timestep: Sensor readings → LSTM cell → Hidden/Cell state")
print("   - Cell state flows through time (long-term memory)")
print("   - Hidden state flows through time (short-term memory)")
print("   - Final hidden state used to predict RUL")
print("   - This allows LSTM to remember degradation patterns from many cycles ago!")

In [ ]:
def create_sequences(data, sequence_length=30):
    """
    Create sequences for LSTM training from engine data
    
    Parameters:
    -----------
    data : DataFrame
        Training data with sensor readings and RUL
    sequence_length : int
        Number of timesteps (cycles) to use for prediction
    
    Returns:
    --------
    X_seq : array, shape (n_sequences, sequence_length, n_features)
        Input sequences (sensor readings over time)
    y_seq : array, shape (n_sequences,)
        Target RUL values
    """
    sequences = []
    targets = []
    
    # Select features (sensors + operational settings)
    feature_cols = op_settings + sensors
    
    # For each engine, create sequences
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences using sliding window
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])  # RUL at end of sequence
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30  # Use 30 cycles to predict RUL
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

print("✅ Sequences created from NASA turbofan engine data!")
print(f"\\n📊 Sequence Statistics:")
print(f"   - Number of sequences: {X_train_seq.shape[0]:,}")
print(f"   - Sequence length: {X_train_seq.shape[1]} cycles")
print(f"   - Features per timestep: {X_train_seq.shape[2]}")
print(f"   - Data shape: (sequences, timesteps, features) = {X_train_seq.shape}")
print(f"   - Target shape: {y_train_seq.shape}")
print(f"   - RUL range: [{y_train_seq.min()}, {y_train_seq.max()}] cycles")

## Step 6: Scale Features and Split Data

In [ ]:
# Scale the features (crucial for LSTMs!)
n_samples, n_timesteps, n_features = X_train_seq.shape
X_reshaped = X_train_seq.reshape(-1, n_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

# Reshape back to sequences
X_train_seq_scaled = X_scaled.reshape(n_samples, n_timesteps, n_features)

# Split into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_seq_scaled, y_train_seq, test_size=0.2, random_state=42
)

print("✅ Features scaled and data split!")
print(f"\\n📊 Final Data Shapes:")
print(f"   - Training sequences: {X_train_split.shape[0]:,}")
print(f"   - Validation sequences: {X_val_split.shape[0]:,}")
print(f"   - Sequence length: {n_timesteps} cycles")
print(f"   - Features per timestep: {n_features}")

## Step 7: Model Design 1 - Single Layer LSTM

Let's start with a simple single-layer LSTM architecture.

In [ ]:
# Model Design 1: Single Layer LSTM
model_1_lstm = Sequential([
    LSTM(128, activation='tanh', return_sequences=False, 
         input_shape=(n_timesteps, n_features), name='lstm_layer_1'),
    Dense(64, activation='relu', name='dense_layer_1'),
    Dense(1, name='output_layer')  # Output: RUL
])

model_1_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("📊 Model Design 1: Single Layer LSTM")
print("=" * 60)
model_1_lstm.summary()

print("\\n💡 Architecture:")
print("   - 1 LSTM layer (128 units)")
print("   - 2 Dense layers (64 → 1)")
print("   - Simple and fast")
print("   - Good starting point")

In [ ]:
# Train Model 1
print("🚀 Training Model 1: Single Layer LSTM...")
print("=" * 60)

history_1_lstm = model_1_lstm.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 1
y_pred_1_lstm = model_1_lstm.predict(X_val_split, verbose=0)

mse_1_lstm = mean_squared_error(y_val_split, y_pred_1_lstm)
mae_1_lstm = mean_absolute_error(y_val_split, y_pred_1_lstm)
rmse_1_lstm = np.sqrt(mse_1_lstm)
r2_1_lstm = r2_score(y_val_split, y_pred_1_lstm)

print("📊 Model 1 Performance (Single Layer LSTM):")
print("=" * 60)
print(f"MSE:  {mse_1_lstm:.2f}")
print(f"MAE:  {mae_1_lstm:.2f} cycles")
print(f"RMSE: {rmse_1_lstm:.2f} cycles")
print(f"R²:   {r2_1_lstm:.4f}")

## Step 8: Model Design 2 - Two Layer LSTM (Stacked)

Stacked LSTMs can learn more complex patterns!

In [ ]:
# Model Design 2: Two Layer Stacked LSTM
model_2_lstm = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, 
         input_shape=(n_timesteps, n_features), name='lstm_layer_1'),
    Dropout(0.2, name='dropout_1'),
    LSTM(64, activation='tanh', return_sequences=False, name='lstm_layer_2'),
    Dropout(0.2, name='dropout_2'),
    Dense(64, activation='relu', name='dense_layer_1'),
    Dense(1, name='output_layer')  # Output: RUL
])

model_2_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("📊 Model Design 2: Two Layer Stacked LSTM")
print("=" * 60)
model_2_lstm.summary()

print("\\n💡 Architecture:")
print("   - 2 LSTM layers (128 → 64 units)")
print("   - return_sequences=True for first layer")
print("   - Dropout layers to prevent overfitting")
print("   - More capacity to learn complex patterns")

In [ ]:
# Train Model 2
print("🚀 Training Model 2: Two Layer Stacked LSTM...")
print("=" * 60)

history_2_lstm = model_2_lstm.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 2
y_pred_2_lstm = model_2_lstm.predict(X_val_split, verbose=0)

mse_2_lstm = mean_squared_error(y_val_split, y_pred_2_lstm)
mae_2_lstm = mean_absolute_error(y_val_split, y_pred_2_lstm)
rmse_2_lstm = np.sqrt(mse_2_lstm)
r2_2_lstm = r2_score(y_val_split, y_pred_2_lstm)

print("📊 Model 2 Performance (Two Layer Stacked LSTM):")
print("=" * 60)
print(f"MSE:  {mse_2_lstm:.2f}")
print(f"MAE:  {mae_2_lstm:.2f} cycles")
print(f"RMSE: {rmse_2_lstm:.2f} cycles")
print(f"R²:   {r2_2_lstm:.4f}")

## Step 9: Model Design 3 - Three Layer Deep LSTM

Deep LSTMs for even more complex patterns!

In [ ]:
# Model Design 3: Three Layer Deep LSTM
model_3_lstm = Sequential([
    LSTM(256, activation='tanh', return_sequences=True, 
         input_shape=(n_timesteps, n_features), name='lstm_layer_1'),
    Dropout(0.2, name='dropout_1'),
    LSTM(128, activation='tanh', return_sequences=True, name='lstm_layer_2'),
    Dropout(0.2, name='dropout_2'),
    LSTM(64, activation='tanh', return_sequences=False, name='lstm_layer_3'),
    Dropout(0.2, name='dropout_3'),
    Dense(64, activation='relu', name='dense_layer_1'),
    Dense(32, activation='relu', name='dense_layer_2'),
    Dense(1, name='output_layer')  # Output: RUL
])

model_3_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("📊 Model Design 3: Three Layer Deep LSTM")
print("=" * 60)
model_3_lstm.summary()

print("\\n💡 Architecture:")
print("   - 3 LSTM layers (256 → 128 → 64 units)")
print("   - Decreasing units in each layer")
print("   - Multiple dropout layers")
print("   - Maximum capacity for complex patterns")

In [ ]:
# Train Model 3
print("🚀 Training Model 3: Three Layer Deep LSTM...")
print("=" * 60)

history_3_lstm = model_3_lstm.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 3
y_pred_3_lstm = model_3_lstm.predict(X_val_split, verbose=0)

mse_3_lstm = mean_squared_error(y_val_split, y_pred_3_lstm)
mae_3_lstm = mean_absolute_error(y_val_split, y_pred_3_lstm)
rmse_3_lstm = np.sqrt(mse_3_lstm)
r2_3_lstm = r2_score(y_val_split, y_pred_3_lstm)

print("📊 Model 3 Performance (Three Layer Deep LSTM):")
print("=" * 60)
print(f"MSE:  {mse_3_lstm:.2f}")
print(f"MAE:  {mae_3_lstm:.2f} cycles")
print(f"RMSE: {rmse_3_lstm:.2f} cycles")
print(f"R²:   {r2_3_lstm:.4f}")

## Step 10: Model Design 4 - Bidirectional LSTM

Bidirectional LSTMs process sequences in both directions!

In [ ]:
# Model Design 4: Bidirectional LSTM
model_4_bilstm = Sequential([
    Bidirectional(LSTM(128, activation='tanh', return_sequences=False), 
                  input_shape=(n_timesteps, n_features), name='bilstm_layer_1'),
    Dropout(0.2, name='dropout_1'),
    Dense(64, activation='relu', name='dense_layer_1'),
    Dense(1, name='output_layer')  # Output: RUL
])

model_4_bilstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("📊 Model Design 4: Bidirectional LSTM")
print("=" * 60)
model_4_bilstm.summary()

print("\\n💡 Architecture:")
print("   - Bidirectional LSTM (128 units each direction = 256 total)")
print("   - Processes sequence forward AND backward")
print("   - Can capture patterns from both directions")
print("   - Useful when context from future timesteps helps")

In [ ]:
# Train Model 4
print("🚀 Training Model 4: Bidirectional LSTM...")
print("=" * 60)

history_4_bilstm = model_4_bilstm.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 4
y_pred_4_bilstm = model_4_bilstm.predict(X_val_split, verbose=0)

mse_4_bilstm = mean_squared_error(y_val_split, y_pred_4_bilstm)
mae_4_bilstm = mean_absolute_error(y_val_split, y_pred_4_bilstm)
rmse_4_bilstm = np.sqrt(mse_4_bilstm)
r2_4_bilstm = r2_score(y_val_split, y_pred_4_bilstm)

print("📊 Model 4 Performance (Bidirectional LSTM):")
print("=" * 60)
print(f"MSE:  {mse_4_bilstm:.2f}")
print(f"MAE:  {mae_4_bilstm:.2f} cycles")
print(f"RMSE: {rmse_4_bilstm:.2f} cycles")
print(f"R²:   {r2_4_bilstm:.4f}")

## Step 11: Model Design 5 - Bidirectional Stacked LSTM

Combine bidirectional and stacked architectures!

In [ ]:
# Model Design 5: Bidirectional Stacked LSTM
model_5_bilstm_stacked = Sequential([
    Bidirectional(LSTM(128, activation='tanh', return_sequences=True), 
                  input_shape=(n_timesteps, n_features), name='bilstm_layer_1'),
    Dropout(0.2, name='dropout_1'),
    Bidirectional(LSTM(64, activation='tanh', return_sequences=False), name='bilstm_layer_2'),
    Dropout(0.2, name='dropout_2'),
    Dense(64, activation='relu', name='dense_layer_1'),
    Dense(1, name='output_layer')  # Output: RUL
])

model_5_bilstm_stacked.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("📊 Model Design 5: Bidirectional Stacked LSTM")
print("=" * 60)
model_5_bilstm_stacked.summary()

print("\\n💡 Architecture:")
print("   - 2 Bidirectional LSTM layers (128 → 64 units each direction)")
print("   - Combines bidirectional and stacked approaches")
print("   - Maximum pattern recognition capability")
print("   - Most complex architecture")

In [ ]:
# Train Model 5
print("🚀 Training Model 5: Bidirectional Stacked LSTM...")
print("=" * 60)

history_5_bilstm_stacked = model_5_bilstm_stacked.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 5
y_pred_5_bilstm_stacked = model_5_bilstm_stacked.predict(X_val_split, verbose=0)

mse_5_bilstm_stacked = mean_squared_error(y_val_split, y_pred_5_bilstm_stacked)
mae_5_bilstm_stacked = mean_absolute_error(y_val_split, y_pred_5_bilstm_stacked)
rmse_5_bilstm_stacked = np.sqrt(mse_5_bilstm_stacked)
r2_5_bilstm_stacked = r2_score(y_val_split, y_pred_5_bilstm_stacked)

print("📊 Model 5 Performance (Bidirectional Stacked LSTM):")
print("=" * 60)
print(f"MSE:  {mse_5_bilstm_stacked:.2f}")
print(f"MAE:  {mae_5_bilstm_stacked:.2f} cycles")
print(f"RMSE: {rmse_5_bilstm_stacked:.2f} cycles")
print(f"R²:   {r2_5_bilstm_stacked:.4f}")

## Step 12: Compare All LSTM Model Designs

Let's compare all 5 model designs!

In [ ]:
# Create comprehensive comparison
comparison_data = {
    'Model': [
        '1-Layer LSTM',
        '2-Layer Stacked LSTM',
        '3-Layer Deep LSTM',
        'Bidirectional LSTM',
        'Bidirectional Stacked LSTM'
    ],
    'MSE': [mse_1_lstm, mse_2_lstm, mse_3_lstm, mse_4_bilstm, mse_5_bilstm_stacked],
    'MAE (cycles)': [mae_1_lstm, mae_2_lstm, mae_3_lstm, mae_4_bilstm, mae_5_bilstm_stacked],
    'RMSE (cycles)': [rmse_1_lstm, rmse_2_lstm, rmse_3_lstm, rmse_4_bilstm, rmse_5_bilstm_stacked],
    'R²': [r2_1_lstm, r2_2_lstm, r2_3_lstm, r2_4_bilstm, r2_5_bilstm_stacked]
}

comparison_df = pd.DataFrame(comparison_data)

print("📊 Comprehensive LSTM Model Comparison (Validation Set):")
print("=" * 100)
print(comparison_df.to_string(index=False))

# Find best model
best_model_idx = comparison_df['R²'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
print(f"\\n🏆 Best Model: {best_model_name}")
print(f"   - R²: {comparison_df.loc[best_model_idx, 'R²']:.4f}")
print(f"   - RMSE: {comparison_df.loc[best_model_idx, 'RMSE (cycles)']:.2f} cycles")
print(f"   - MAE: {comparison_df.loc[best_model_idx, 'MAE (cycles)']:.2f} cycles")

## Step 13: Visualize Model Comparison

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('LSTM Model Designs Comparison for RUL Prediction', fontsize=16, fontweight='bold')

models = comparison_df['Model'].values
colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

# R² comparison
axes[0, 0].bar(range(len(models)), comparison_df['R²'], color=colors, alpha=0.7, 
               edgecolor='black', linewidth=1.5)
axes[0, 0].set_xticks(range(len(models)))
axes[0, 0].set_xticklabels([m.replace(' LSTM', '') for m in models], rotation=45, ha='right')
axes[0, 0].set_ylabel('R² Score', fontsize=12, fontweight='bold')
axes[0, 0].set_title('R² Score Comparison', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['R²']):
    axes[0, 0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# RMSE comparison
axes[0, 1].bar(range(len(models)), comparison_df['RMSE (cycles)'], color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(range(len(models)))
axes[0, 1].set_xticklabels([m.replace(' LSTM', '') for m in models], rotation=45, ha='right')
axes[0, 1].set_ylabel('RMSE (cycles)', fontsize=12, fontweight='bold')
axes[0, 1].set_title('RMSE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['RMSE (cycles)']):
    axes[0, 1].text(i, v + max(comparison_df['RMSE (cycles)'])*0.02, f'{v:.2f}', 
                    ha='center', fontweight='bold', fontsize=9)

# MAE comparison
axes[1, 0].bar(range(len(models)), comparison_df['MAE (cycles)'], color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(range(len(models)))
axes[1, 0].set_xticklabels([m.replace(' LSTM', '') for m in models], rotation=45, ha='right')
axes[1, 0].set_ylabel('MAE (cycles)', fontsize=12, fontweight='bold')
axes[1, 0].set_title('MAE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['MAE (cycles)']):
    axes[1, 0].text(i, v + max(comparison_df['MAE (cycles)'])*0.02, f'{v:.2f}', 
                    ha='center', fontweight='bold', fontsize=9)

# Parameter counts
def count_params(model):
    return model.count_params()

param_counts = [
    count_params(model_1_lstm),
    count_params(model_2_lstm),
    count_params(model_3_lstm),
    count_params(model_4_bilstm),
    count_params(model_5_bilstm_stacked)
]

axes[1, 1].bar(range(len(models)), param_counts, color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[1, 1].set_xticks(range(len(models)))
axes[1, 1].set_xticklabels([m.replace(' LSTM', '') for m in models], rotation=45, ha='right')
axes[1, 1].set_ylabel('Number of Parameters', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Model Complexity (Parameter Count)', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(param_counts):
    axes[1, 1].text(i, v + max(param_counts)*0.01, f'{v:,}', ha='center', 
                   fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Model comparison visualization created!")

In [ ]:
# Plot training history for all models
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('LSTM Training Progress: All Model Designs', fontsize=16, fontweight='bold')

histories = [history_1_lstm, history_2_lstm, history_3_lstm, history_4_bilstm, history_5_bilstm_stacked]
labels = ['1-Layer', '2-Layer Stacked', '3-Layer Deep', 'Bidirectional', 'Bidirectional Stacked']
colors_plot = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

# Loss comparison
for hist, label, color in zip(histories, labels, colors_plot):
    axes[0].plot(hist.history['loss'], label=f'{label} (Train)', linestyle='-', linewidth=2, color=color)
    axes[0].plot(hist.history['val_loss'], label=f'{label} (Val)', linestyle='--', linewidth=2, color=color, alpha=0.7)

axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_title('Loss Comparison', fontsize=13, fontweight='bold')
axes[0].legend(ncol=2, fontsize=9)
axes[0].grid(True, alpha=0.3)

# MAE comparison
for hist, label, color in zip(histories, labels, colors_plot):
    axes[1].plot(hist.history['mae'], label=f'{label} (Train)', linestyle='-', linewidth=2, color=color)
    axes[1].plot(hist.history['val_mae'], label=f'{label} (Val)', linestyle='--', linewidth=2, color=color, alpha=0.7)

axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE (cycles)', fontsize=12, fontweight='bold')
axes[1].set_title('MAE Comparison', fontsize=13, fontweight='bold')
axes[1].legend(ncol=2, fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Training progress visualization created!")

## Step 15: Visualize Predictions for All Models

In [ ]:
# Collect all predictions
predictions = {
    '1-Layer LSTM': y_pred_1_lstm,
    '2-Layer Stacked': y_pred_2_lstm,
    '3-Layer Deep': y_pred_3_lstm,
    'Bidirectional': y_pred_4_bilstm,
    'Bidirectional Stacked': y_pred_5_bilstm_stacked
}

# Plot predictions vs actual for all models
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('RUL Predictions vs Actual Values - All LSTM Models', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, (name, pred) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.scatter(y_val_split, pred, alpha=0.5, s=20, color=colors[idx])
    min_val = min(y_val_split.min(), pred.min())
    max_val = max(y_val_split.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual RUL (cycles)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted RUL (cycles)', fontsize=11, fontweight='bold')
    r2_val = comparison_df[comparison_df['Model'] == name]['R²'].values[0]
    ax.set_title(f'{name}\\n(R² = {r2_val:.4f})', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove last subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

print("✅ Predictions visualization created!")

## Step 16: Evaluate Best Model on Test Set

Let's test the best model on unseen engines!

In [ ]:
# Create test sequences
def create_test_sequences(data, sequence_length=30):
    """Create test sequences (last sequence_length timesteps for each engine)"""
    sequences = []
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        # Get last sequence_length timesteps
        if len(unit_data) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            # Pad with first value if sequence is too short
            padding = np.tile(unit_features[0:1], (sequence_length - len(unit_data), 1))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

# Create test sequences
X_test_seq = create_test_sequences(test_df, sequence_length)

# Scale test sequences
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape[0], n_timesteps, n_features)

# Get actual RUL from the RUL file
y_test_actual = rul_df['RUL'].values

print("✅ Test data prepared!")
print(f"   - Test sequences: {X_test_seq_scaled.shape[0]}")
print(f"   - Actual RUL range: [{y_test_actual.min()}, {y_test_actual.max()}] cycles")

# Use best model based on validation performance
best_models = {
    '1-Layer LSTM': model_1_lstm,
    '2-Layer Stacked LSTM': model_2_lstm,
    '3-Layer Deep LSTM': model_3_lstm,
    'Bidirectional LSTM': model_4_bilstm,
    'Bidirectional Stacked LSTM': model_5_bilstm_stacked
}

best_model = best_models[best_model_name]
y_test_pred = best_model.predict(X_test_seq_scaled, verbose=0)

# Calculate metrics on test set
mse_test = mean_squared_error(y_test_actual, y_test_pred)
mae_test = mean_absolute_error(y_test_actual, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_actual, y_test_pred)

print(f"\\n📊 Best Model ({best_model_name}) Performance on Test Set:")
print("=" * 60)
print(f"MSE:  {mse_test:.2f}")
print(f"MAE:  {mae_test:.2f} cycles")
print(f"RMSE: {rmse_test:.2f} cycles")
print(f"R²:   {r2_test:.4f}")

print(f"\\n💡 Interpretation:")
print(f"   - On average, predictions are off by {mae_test:.2f} cycles")
print(f"   - The model explains {r2_test*100:.1f}% of the variance in RUL")
print(f"   - This is performance on completely unseen engines!")

In [ ]:
# Plot test set predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Test Set RUL Predictions - {best_model_name}', fontsize=16, fontweight='bold')

# Scatter plot: Predictions vs Actual
axes[0].scatter(y_test_actual, y_test_pred, alpha=0.6, s=50, color='blue')
min_val = min(y_test_actual.min(), y_test_pred.min())
max_val = max(y_test_actual.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_title(f'Predictions vs Actual (R² = {r2_test:.4f})', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error distribution
errors = y_test_actual.flatten() - y_test_pred.flatten()
axes[1].hist(errors, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1].axvline(x=mae_test, color='blue', linestyle='--', linewidth=2, label=f'MAE = {mae_test:.2f}')
axes[1].axvline(x=-mae_test, color='blue', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (cycles)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1].set_title('Error Distribution', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Test set visualization created!")

## 🎓 Summary and Key Takeaways

### ✅ What You've Learned:

1. **LSTM Architecture**:
   - Forget Gate: Controls what to forget from cell state
   - Input Gate: Controls what new information to store
   - Output Gate: Controls what to output
   - Cell State: Long-term memory
   - Hidden State: Short-term memory/output

2. **LSTM Model Designs**:
   - **1-Layer LSTM**: Simple, fast, good baseline
   - **2-Layer Stacked**: More capacity, learns complex patterns
   - **3-Layer Deep**: Maximum capacity, most complex
   - **Bidirectional**: Processes sequences both ways
   - **Bidirectional Stacked**: Combines bidirectional + stacked

3. **Key Insights**:
   - More layers ≠ Always better performance
   - Bidirectional can help but increases complexity
   - Dropout prevents overfitting in deeper models
   - Model selection depends on data and problem

### 💡 Important Findings:

- **Architecture Matters**: Different designs have different strengths
- **Complexity vs Performance**: More complex doesn't always mean better
- **Regularization**: Dropout is crucial for deeper models
- **Bidirectional**: Can help when context from both directions matters

### 📚 LSTM Design Guidelines:

| Design | Best For | Complexity | Parameters |
|--------|----------|------------|------------|
| **1-Layer** | Simple problems, baseline | Low | Fewest |
| **2-Layer Stacked** | Complex patterns | Medium | Medium |
| **3-Layer Deep** | Very complex patterns | High | High |
| **Bidirectional** | When future context helps | Medium-High | High |
| **Bidirectional Stacked** | Maximum capacity | Highest | Highest |

### 🔧 Best Practices:

1. **Start Simple**: Begin with 1-layer LSTM
2. **Add Complexity Gradually**: Try 2-layer, then 3-layer if needed
3. **Use Dropout**: Essential for deeper models
4. **Consider Bidirectional**: If context from both directions helps
5. **Early Stopping**: Monitor validation loss
6. **Regularization**: Dropout prevents overfitting

### 🆚 Model Design Comparison:

| Aspect | 1-Layer | 2-Layer | 3-Layer | Bidirectional | Bi+Stacked |
|--------|---------|---------|---------|---------------|------------|
| **Speed** | Fastest | Medium | Slow | Medium | Slowest |
| **Capacity** | Low | Medium | High | Medium-High | Highest |
| **Overfitting Risk** | Low | Medium | High | Medium | Highest |
| **Best For** | Baseline | Most problems | Complex patterns | Bidirectional context | Maximum capacity |

---

**Great job exploring different LSTM architectures for RUL prediction! 🧠✈️✨**